# 05 — SCD Type 2: Train Schedule History Dimension

**Slowly Changing Dimension Type 2 — Full History**

Used for attributes where historical snapshots matter for **trend analysis and delay modelling**:
- Schedule time changes (departure/arrival)
- Route modifications (stops added/removed)
- Zone handoffs between operators
- Train name changes

**Strategy:**
1. Detect rows where tracked attributes changed
2. Expire old rows: set `is_current = 'N'`, `eff_to_date = today - 1`
3. Insert new rows: `is_current = 'Y'`, new surrogate key, `eff_from_date = today`

**Target table:** `gold.dim_train_schedule_scd2`

> **Why SCD2 here?** The Silver layer has 25.8M delay records timestamped across years.
> Point-in-time joins (`fact.event_date BETWEEN dim.eff_from_date AND dim.eff_to_date`)
> ensure each delay record is matched to the schedule that was *active at that moment* —
> not today's schedule. This is critical for accurate delay attribution.

In [0]:
# ─────────────────────────────────────────────
# 0. Imports & Spark session
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_date, lit, expr, upper, trim,
    when, coalesce, to_date
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType
)
from delta.tables import DeltaTable
import uuid

spark = SparkSession.builder \
    .appName('IndianRailways_SCD2_TrainSchedule') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .getOrCreate()

spark.conf.set('spark.sql.shuffle.partitions', '200')

HIGH_DATE = '9999-12-31'  # Sentinel for open-ended current rows
print('Spark ready. High date sentinel:', HIGH_DATE)

Spark ready. High date sentinel: 9999-12-31


In [0]:
TARGET_TABLE_SCD2 = 'gold.dim_train_schedule_scd2'

print(f"Ensuring table {TARGET_TABLE_SCD2} exists...")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TARGET_TABLE_SCD2} (
    surrogate_key       STRING  NOT NULL COMMENT 'UUID — unique per train version',
    train_no            STRING  NOT NULL COMMENT 'Natural business key',
    train_name          STRING,
    zone                STRING,
    origin_station      STRING,
    dest_station        STRING,
    scheduled_dep       STRING  COMMENT 'HH:MM origin departure',
    scheduled_arr       STRING  COMMENT 'HH:MM destination arrival',
    journey_duration_h  DOUBLE,
    route_stops         INT     COMMENT 'Number of intermediate halts',
    distance_km         INT,
    frequency           STRING,
    eff_from_date       DATE    NOT NULL COMMENT 'Version valid from (inclusive)',
    eff_to_date         DATE    NOT NULL COMMENT '9999-12-31 for current row',
    is_current          STRING  NOT NULL COMMENT 'Y = active | N = expired',
    change_reason       STRING  COMMENT 'Human-readable reason for version creation'
)
USING DELTA
PARTITIONED BY (zone)
COMMENT 'SCD Type 2 — full schedule history; enables point-in-time delay attribution'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'delta.enableChangeDataFeed'       = 'true'
)
''')
print(f'Table ready: {TARGET_TABLE_SCD2}')

Ensuring table gold.dim_train_schedule_scd2 exists...
Table ready: gold.dim_train_schedule_scd2


In [0]:
# ─────────────────────────────────────────────
# 2. Bootstrap: load initial historical versions
#    (Run ONCE on first deployment)
# ─────────────────────────────────────────────
initial_records = [
    # surrogate_key           train_no  train_name                     zone   origin dest  dep     arr      dur   stops  km    freq        eff_from      eff_to        cur  reason
    ('TRN-22691-V1-2022',   '22691', 'Rajdhani Express',          'CR',  'SBC', 'NDLS','20:15','07:55',35.7, 22,  2444, 'Daily',    '2022-01-01', '2024-05-31', 'N', 'Initial load — CR zone'),
    ('TRN-22691-V2-2024',   '22691', 'Rajdhani Express',          'SWR', 'SBC', 'NDLS','20:15','07:55',35.7, 22,  2444, 'Daily',    '2024-06-01', '9999-12-31', 'Y', 'Zone handoff CR→SWR'),
    ('TRN-12301-V1-2019',   '12301', 'Howrah Rajdhani Express',   'ER',  'HWH', 'NDLS','13:55','10:05',20.2, 6,   1441, 'Daily',    '2019-06-01', '2023-11-30', 'N', 'Initial load'),
    ('TRN-12301-V2-2023',   '12301', 'Howrah Rajdhani Express',   'ER',  'HWH', 'NDLS','13:55','09:55',20.0, 5,   1441, 'Daily',    '2023-12-01', '9999-12-31', 'Y', 'Schedule optimised -10min, 1 stop removed'),
    ('TRN-12002-V1-2015',   '12002', 'Bhopal Shatabdi Express',   'NCR', 'NDLS','BPL', '06:00','14:00',8.0,  4,   704,  'Daily',    '2015-01-01', '2025-12-31', 'N', 'Initial load — ICF rake'),
    ('TRN-12002-V2-2026',   '12002', 'New Delhi Shatabdi',        'NCR', 'NDLS','BPL', '06:00','13:45',7.75, 4,   704,  'Daily',    '2026-01-01', '9999-12-31', 'Y', 'Vande Bharat upgrade + rename'),
    ('TRN-12951-V1-2010',   '12951', 'Mumbai Rajdhani Express',   'WR',  'MMCT','NDLS','17:00','08:35',15.6, 4,   1384, 'Daily',    '2010-01-01', '2025-11-11', 'N', 'Initial load'),
    ('TRN-12951-V2-2025',   '12951', 'Mumbai Rajdhani Express',   'WR',  'MMCT','NDLS','16:35','08:35',16.0, 4,   1384, 'Daily',    '2025-11-12', '9999-12-31', 'Y', 'Departure revised -25min'),
    ('TRN-12621-V1-2018',   '12621', 'Tamil Nadu Express',        'SR',  'MAS', 'NDLS','22:00','07:10',33.2, 13,  2184, 'Daily',    '2018-01-01', '9999-12-31', 'Y', 'Initial load'),
    ('TRN-20504-V1-2023',   '20504', 'Vande Bharat Express',      'NR',  'NDLS','SVDK','06:00','11:30',5.5,  3,   655,  'Daily',    '2023-04-01', '9999-12-31', 'Y', 'New service launch'),
    ('TRN-12627-V1-2020',   '12627', 'Karnataka Express',         'SWR', 'SBC', 'NDLS','19:30','10:30',39.0, 35,  3075, 'Daily',    '2020-01-01', '9999-12-31', 'Y', 'Initial load'),
    ('TRN-12649-V1-2021',   '12649', 'Karnataka Sampark Kranti',  'SWR', 'YPR', 'NDLS','21:40','07:10',33.5, 18,  2628, 'BiWeekly', '2021-03-01', '9999-12-31', 'Y', 'Initial load'),
    ('TRN-12309-V1-2019',   '12309', 'Rajendra Nagar Express',    'ECR', 'RJPB','NDLS','08:50','12:20',27.5, 14,  1005, 'Daily',    '2019-01-01', '2024-06-30', 'N', 'Initial load'),
    ('TRN-12309-V2-2024',   '12309', 'Rajendra Nagar Express',    'ECR', 'RJPB','NDLS','08:50','12:20',27.5, 17,  1085, 'Daily',    '2024-07-01', '9999-12-31', 'Y', 'Route extended — 3 new stops'),
    ('TRN-12013-V1-2022',   '12013', 'Amritsar Shatabdi',         'NR',  'NDLS','ASR', '07:20','13:10',5.83, 5,   449,  'Daily',    '2022-01-01', '9999-12-31', 'Y', 'Initial load'),
    ('TRN-12025-V1-2023',   '12025', 'Pune Shatabdi',             'CR',  'PUNE','MMCT','07:10','09:30',2.33, 2,   192,  'Daily',    '2023-06-01', '9999-12-31', 'Y', 'Initial load'),
]

bootstrap_schema = StructType([
    StructField('surrogate_key',      StringType(), False),
    StructField('train_no',           StringType(), False),
    StructField('train_name',         StringType(), True),
    StructField('zone',               StringType(), True),
    StructField('origin_station',     StringType(), True),
    StructField('dest_station',       StringType(), True),
    StructField('scheduled_dep',      StringType(), True),
    StructField('scheduled_arr',      StringType(), True),
    StructField('journey_duration_h', StringType(), True),
    StructField('route_stops',        IntegerType(),True),
    StructField('distance_km',        IntegerType(),True),
    StructField('frequency',          StringType(), True),
    StructField('eff_from_date',      StringType(), True),
    StructField('eff_to_date',        StringType(), True),
    StructField('is_current',         StringType(), False),
    StructField('change_reason',      StringType(), True),
])

bootstrap_df = spark.createDataFrame(initial_records, schema=bootstrap_schema) \
    .withColumn('eff_from_date',      to_date(col('eff_from_date'))) \
    .withColumn('eff_to_date',        to_date(col('eff_to_date'))) \
    .withColumn('journey_duration_h', col('journey_duration_h').cast('double'))

# Only write if table is empty (bootstrap guard)
existing_count = spark.table(TARGET_TABLE).count()
if existing_count == 0:
    bootstrap_df.write.format('delta').mode('append').saveAsTable(TARGET_TABLE)
    print(f'Bootstrap complete — {bootstrap_df.count()} rows loaded.')
else:
    print(f'Table already has {existing_count} rows — skipping bootstrap.')

Table already has 16 rows — skipping bootstrap.


In [0]:
# ─────────────────────────────────────────────
# 3. Incoming source data (new schedule batch)
#    Replace with your live NTES feed / CSV read
# ─────────────────────────────────────────────
new_schedules = [
    # Simulate a schedule change: Tamil Nadu Express gets +2 stops and new dep time
    ('12621', 'Tamil Nadu Express',       'SR',  'MAS', 'NDLS', '21:45', '07:10', 33.4, 15, 2184, 'Daily'),
    # Rajdhani Express — no change (should NOT create a new SCD2 row)
    ('22691', 'Rajdhani Express',         'SWR', 'SBC', 'NDLS', '20:15', '07:55', 35.7, 22, 2444, 'Daily'),
    # Vande Bharat — frequency change BiWeekly→Daily
    ('20504', 'Vande Bharat Express',     'NR',  'NDLS','SVDK', '06:00', '11:30', 5.5,  3,  655,  'BiWeekly'),
]

incoming_schema = StructType([
    StructField('train_no',           StringType(),  False),
    StructField('train_name',         StringType(),  True),
    StructField('zone',               StringType(),  True),
    StructField('origin_station',     StringType(),  True),
    StructField('dest_station',       StringType(),  True),
    StructField('scheduled_dep',      StringType(),  True),
    StructField('scheduled_arr',      StringType(),  True),
    StructField('journey_duration_h', StringType(),  True),
    StructField('route_stops',        IntegerType(), True),
    StructField('distance_km',        IntegerType(), True),
    StructField('frequency',          StringType(),  True),
])

source_df = spark.createDataFrame(new_schedules, schema=incoming_schema) \
    .withColumn('journey_duration_h', col('journey_duration_h').cast('double'))

print('Incoming records:', source_df.count())
source_df.show(truncate=False)

Incoming records: 3
+--------+--------------------+----+--------------+------------+-------------+-------------+------------------+-----------+-----------+---------+
|train_no|train_name          |zone|origin_station|dest_station|scheduled_dep|scheduled_arr|journey_duration_h|route_stops|distance_km|frequency|
+--------+--------------------+----+--------------+------------+-------------+-------------+------------------+-----------+-----------+---------+
|12621   |Tamil Nadu Express  |SR  |MAS           |NDLS        |21:45        |07:10        |33.4              |15         |2184       |Daily    |
|22691   |Rajdhani Express    |SWR |SBC           |NDLS        |20:15        |07:55        |35.7              |22         |2444       |Daily    |
|20504   |Vande Bharat Express|NR  |NDLS          |SVDK        |06:00        |11:30        |5.5               |3          |655        |BiWeekly |
+--------+--------------------+----+--------------+------------+-------------+-------------+------------

In [0]:
# ─────────────────────────────────────────────
# 4. Detect changed rows
#    Join incoming vs current active rows
# ─────────────────────────────────────────────
current_dim = spark.table(TARGET_TABLE).filter(col('is_current') == 'Y')

changed_df = source_df.alias('src').join(
    current_dim.alias('dim'),
    on='train_no',
    how='inner'
).filter(
    (col('src.scheduled_dep')      != col('dim.scheduled_dep'))      |
    (col('src.scheduled_arr')      != col('dim.scheduled_arr'))      |
    (col('src.route_stops')        != col('dim.route_stops'))        |
    (col('src.zone')               != col('dim.zone'))               |
    (col('src.frequency')          != col('dim.frequency'))
).select(
    col('src.train_no'),
    col('src.train_name'),
    col('src.zone'),
    col('src.origin_station'),
    col('src.dest_station'),
    col('src.scheduled_dep'),
    col('src.scheduled_arr'),
    col('src.journey_duration_h'),
    col('src.route_stops'),
    col('src.distance_km'),
    col('src.frequency'),
)

changed_count = changed_df.count()
print(f'Changed trains detected: {changed_count}')
changed_df.show(truncate=False)

Changed trains detected: 2
+--------+--------------------+----+--------------+------------+-------------+-------------+------------------+-----------+-----------+---------+
|train_no|train_name          |zone|origin_station|dest_station|scheduled_dep|scheduled_arr|journey_duration_h|route_stops|distance_km|frequency|
+--------+--------------------+----+--------------+------------+-------------+-------------+------------------+-----------+-----------+---------+
|20504   |Vande Bharat Express|NR  |NDLS          |SVDK        |06:00        |11:30        |5.5               |3          |655        |BiWeekly |
|12621   |Tamil Nadu Express  |SR  |MAS           |NDLS        |21:45        |07:10        |33.4              |15         |2184       |Daily    |
+--------+--------------------+----+--------------+------------+-------------+-------------+------------------+-----------+-----------+---------+



In [0]:
# ─────────────────────────────────────────────
# 5. SCD Type 2 — Step A: Expire old current rows
# ─────────────────────────────────────────────
if changed_count > 0:
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias('target')
        .merge(
            changed_df.alias('source'),
            'target.train_no = source.train_no AND target.is_current = \'Y\''
        )
        .whenMatchedUpdate(
            set={
                'is_current':  "'N'",
                # expire yesterday so new row eff_from = today has no gap/overlap
                'eff_to_date': "date_sub(current_date(), 1)",
            }
        )
        .execute()
    )
    print('Step A complete — old rows expired.')
else:
    print('No changes detected — Step A skipped.')

Step A complete — old rows expired.


In [0]:
# ─────────────────────────────────────────────
# 6. SCD Type 2 — Step B: Insert new current rows
# ─────────────────────────────────────────────
from pyspark.sql.functions import udf
import uuid as _uuid

@udf(StringType())
def gen_uuid():
    return str(_uuid.uuid4())

if changed_count > 0:
    new_rows_df = changed_df \
        .withColumn('surrogate_key',  gen_uuid()) \
        .withColumn('eff_from_date',  current_date()) \
        .withColumn('eff_to_date',    to_date(lit(HIGH_DATE))) \
        .withColumn('is_current',     lit('Y')) \
        .withColumn('change_reason',  lit('SCD2 auto-detected schedule change'))

    new_rows_df.write \
        .format('delta') \
        .mode('append') \
        .option('mergeSchema', 'false') \
        .saveAsTable(TARGET_TABLE)

    print(f'Step B complete — {new_rows_df.count()} new current rows inserted.')
else:
    print('No changes detected — Step B skipped.')

Step B complete — 0 new current rows inserted.


In [0]:
# ─────────────────────────────────────────────
# 7. Validate: view full SCD2 table
# ─────────────────────────────────────────────
final_df = spark.table(TARGET_TABLE)

print('Total rows (all versions):', final_df.count())
print('Current rows (is_current=Y):', final_df.filter("is_current='Y'").count())
print('Historical rows (is_current=N):', final_df.filter("is_current='N'").count())

final_df.orderBy('train_no', 'eff_from_date').select(
    'train_no', 'train_name', 'zone', 'scheduled_dep',
    'route_stops', 'eff_from_date', 'eff_to_date', 'is_current', 'change_reason'
).show(30, truncate=False)

Total rows (all versions): 16
Current rows (is_current=Y): 9
Historical rows (is_current=N): 7
+--------+------------------------+----+-------------+-----------+-------------+-----------+----------+-----------------------------+
|train_no|train_name              |zone|scheduled_dep|route_stops|eff_from_date|eff_to_date|is_current|change_reason                |
+--------+------------------------+----+-------------+-----------+-------------+-----------+----------+-----------------------------+
|12002   |Bhopal Shatabdi Express |NCR |06:00        |4          |2015-01-01   |2025-12-31 |N         |Initial load — ICF rake      |
|12002   |New Delhi Shatabdi      |NCR |06:00        |4          |2026-01-01   |9999-12-31 |Y         |Vande Bharat upgrade + rename|
|12013   |Amritsar Shatabdi       |NR  |07:20        |5          |2022-01-01   |9999-12-31 |Y         |Initial load                 |
|12025   |Pune Shatabdi           |CR  |07:10        |2          |2023-06-01   |9999-12-31 |Y        

In [0]:
from pyspark.sql.functions import col

# ─────────────────────────────────────────────
# 8. Point-in-time join example
#    Join Silver delay records to the schedule
#    that was active ON the day of the delay
# ─────────────────────────────────────────────

# Ensure this points to your SCD2 history table!
TARGET_TABLE_SCD2 = 'gold.dim_train_schedule_scd2'
silver_df = spark.table('silver.fact_train_delays')  

point_in_time_df = silver_df.alias('f').join(
    spark.table(TARGET_TABLE_SCD2).alias('d'),
    on=[
        col('f.train_no') == col('d.train_no'),
        col('f.event_date') >= col('d.eff_from_date'),
        col('f.event_date') <= col('d.eff_to_date'),
    ],
    how='left'
).select(
    'f.train_no', 'f.event_date', 'f.station_code',
    'f.delay_arr_mins',  # FIX: Changed from 'f.delay_mins'
    'd.zone', 'd.route_stops',
    'd.scheduled_dep', 'd.is_current'
)

print('Point-in-time joined records:', point_in_time_df.count())
point_in_time_df.show(10, truncate=False)

Point-in-time joined records: 0
+--------+----------+------------+--------------+----+-----------+-------------+----------+
|train_no|event_date|station_code|delay_arr_mins|zone|route_stops|scheduled_dep|is_current|
+--------+----------+------------+--------------+----+-----------+-------------+----------+
+--------+----------+------------+--------------+----+-----------+-------------+----------+



In [0]:
from delta.tables import DeltaTable

TARGET_TABLE_SCD2 = 'gold.dim_train_schedule_scd2'
delta_target = DeltaTable.forName(spark, TARGET_TABLE_SCD2)

print('\n--- Delta History ---')
# Show the last 10 operations
delta_target.history(10).select(
    'version', 'timestamp', 'operation', 'operationMetrics'
).show(truncate=False)

# FIX: Safely extract the pure integer from the Spark Row
latest_version = int(delta_target.history(1).head().version)

print(f'\n--- Change Data Feed (Showing changes from Version {latest_version}) ---')
spark.read.format('delta') \
    .option('readChangeFeed', 'true') \
    .option('startingVersion', latest_version) \
    .table(TARGET_TABLE_SCD2) \
    .show(20, truncate=False)


--- Delta History ---
+-------+-------------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation   |operationMetrics                                                                                                                                                                                                                  